# 🚀 5 MODEL - Full 7600 Samples

MPNet, BGE-M3, E5-large, Qwen3, Jina v5

## Setup: Option 1 - Clone from GitHub (RECOMMENDED)
Bu yöntem her zaman en güncel kodu alır!

In [ ]:
# GitHub'dan clone et (en güncel kod!)
!git clone https://github.com/YOUR-USERNAME/zeroshot-text-classification-benchmark.git
%cd zeroshot-text-classification-benchmark
!pip install -q -r requirements.txt

## Setup: Option 2 - Use Drive (Manuel Upload)
Eğer Drive'a manuel yüklediyseniz

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir('/content/drive/MyDrive/zero_shot_colab_final')
print("✅", os.getcwd())

In [ ]:
!pip install -q datasets sentence-transformers scikit-learn pandas pyyaml matplotlib seaborn transformers accelerate

In [ ]:
import torch
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"{torch.cuda.get_device_name(0)} - {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Jina v5 Custom Encoder
code = '''\nimport torch\nimport numpy as np\nfrom sentence_transformers import SentenceTransformer\n\nclass JinaV5Encoder:\n    def __init__(self, model_name="jinaai/jina-embeddings-v5-text-nano", task="classification"):\n        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n        self.task = task\n        print(f"Loading Jina v5 on {self.device} (task={task})...")\n        self.model = SentenceTransformer(\n            model_name,\n            trust_remote_code=True,\n            device=self.device,\n            model_kwargs={"dtype": torch.bfloat16}\n        )\n        print("Jina v5 loaded!")\n    \n    def encode(self, texts, batch_size=32, normalize=True, show_progress=False):\n        if isinstance(texts, str):\n            texts = [texts]\n        embeddings = self.model.encode(\n            sentences=texts,\n            task=self.task,\n            batch_size=batch_size,\n            show_progress_bar=show_progress,\n            convert_to_numpy=True\n        )\n        if normalize:\n            import numpy as np\n            embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)\n        return embeddings\n'''

with open('src/encoders_custom.py', 'w') as f:
    f.write(code)
print("✅ Jina v5 encoder")

In [ ]:
# Update main encoder
code = '''\nfrom sentence_transformers import SentenceTransformer\nimport numpy as np\nfrom typing import List, Union\n\nclass BiEncoder:\n    def __init__(self, model_name: str, device: str = None):\n        print(f"Loading: {model_name}")\n        self.model_name = model_name\n        if "jina-embeddings-v5" in model_name:\n            from src.encoders_custom import JinaV5Encoder\n            self.model = JinaV5Encoder(model_name)\n            self.backend = "custom"\n        else:\n            self.model = SentenceTransformer(model_name, device=device)\n            self.backend = "st"\n    \n    def encode(self, texts: Union[str, List[str]], batch_size=32, normalize=True, show_progress=False) -> np.ndarray:\n        if self.backend == "custom":\n            return self.model.encode(texts, batch_size, normalize, show_progress)\n        return self.model.encode(texts, batch_size=batch_size, normalize_embeddings=normalize, show_progress_bar=show_progress, convert_to_numpy=True)\n\ndef cosine_similarity_matrix(a, b):\n    return np.matmul(a, b.T)\n\ndef compute_similarities(text_embeddings, label_embeddings):\n    return cosine_similarity_matrix(text_embeddings, label_embeddings)\n'''

with open('src/encoders.py', 'w') as f:
    f.write(code)
print("✅ Encoder updated")

In [ ]:
import yaml
from pathlib import Path

def create_config(model_name, exp_name):
    config = {
        "experiment_name": exp_name,
        "dataset": {"name": "ag_news", "split": "test", "text_column": "text", "label_column": "label", "max_samples": None},
        "task": {"type": "zero_shot_classification", "label_mode": "description", "language": "en"},
        "models": {"biencoder": {"provider": "hf", "name": model_name}, "reranker": None},
        "pipeline": {"mode": "biencoder", "normalize_embeddings": True},
        "evaluation": {"metrics": ["accuracy", "macro_f1", "per_class_f1"]},
        "output": {"save_predictions": True, "save_metrics": True, "output_dir": "results/raw"}
    }
    Path("experiments/standard").mkdir(parents=True, exist_ok=True)
    with open(f"experiments/standard/{exp_name}.yaml", "w") as f:
        yaml.dump(config, f)
    print(f"✅ {exp_name}")

models = [
    ("sentence-transformers/all-mpnet-base-v2", "mpnet_full"),
    ("BAAI/bge-m3", "bge_full"),
    ("intfloat/multilingual-e5-large", "e5_full"),
    ("Qwen/Qwen3-Embedding-8B", "qwen3_full"),
    ("jinaai/jina-embeddings-v5-text-nano", "jina_v5_full"),
]

for model, name in models:
    create_config(model, name)

In [ ]:
# ⚙️ SETTINGS
SKIP_EXISTING = True  # Set to False to re-run all experiments

experiments = ["mpnet_full", "bge_full", "e5_full", "qwen3_full", "jina_v5_full"]
results = []

if SKIP_EXISTING:
    print("Mode: --skip-existing (will skip if results exist)")
    print("💡 To re-run all experiments, set SKIP_EXISTING = False\n")
else:
    print("Mode: OVERWRITE (will re-run all experiments)\n")

for exp in experiments:
    print(f"\n{'='*70}\n{exp}\n{'='*70}")
    try:
        cmd = f"python main.py --config experiments/standard/{exp}.yaml"
        if SKIP_EXISTING:
            cmd += " --skip-existing"
        !{cmd}
        results.append((exp, "✅"))
    except Exception as e:
        print(f"ERROR: {e}")
        results.append((exp, "❌"))

print("\n" + "="*70)
for exp, status in results:
    print(f"{exp}: {status}")

In [ ]:
import json
import pandas as pd
from pathlib import Path

files = list(Path("results/raw").glob("*_full_metrics.json"))
data = []
for f in files:
    with open(f) as fp:
        m = json.load(fp)
    data.append({"model": m.get("biencoder", "N/A"), "samples": m["num_samples"], "accuracy": m["accuracy"] * 100, "macro_f1": m["macro_f1"] * 100})

df = pd.DataFrame(data).sort_values("macro_f1", ascending=False)
print("\n" + "="*70 + "\nFINAL RESULTS\n" + "="*70)
print(df.to_string(index=False))
df.to_csv("results/FINAL_COMPARISON.csv", index=False)
print("\n✅ results/FINAL_COMPARISON.csv")

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

df_plot = df.sort_values("accuracy", ascending=True)
ax1.barh(range(len(df_plot)), df_plot["accuracy"])
ax1.set_yticks(range(len(df_plot)))
ax1.set_yticklabels(df_plot["model"])
ax1.set_xlabel("Accuracy (%)")
ax1.set_title("Accuracy")

df_plot = df.sort_values("macro_f1", ascending=True)
ax2.barh(range(len(df_plot)), df_plot["macro_f1"])
ax2.set_yticks(range(len(df_plot)))
ax2.set_yticklabels(df_plot["model"])
ax2.set_xlabel("Macro F1 (%)")
ax2.set_title("Macro F1")

plt.tight_layout()
plt.savefig("results/plots/comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ results/plots/comparison.png")

## ✅ 5 MODELS!

- MPNet (420M)
- BGE-M3 (567M)
- E5-large (560M, multilingual)
- Qwen3 (8B)
- Jina v5 nano (33M)